# The Binomial Tree: the simple version
One idea, step by step: **price an option by building a tree of possible stock prices, then working backwards.**

No live data and no extras. Change the numbers in the next cell, then choose *Run All*.
(The bigger tool with live data, Greeks and strategies is `binomial_option_pricer.ipynb`.)

*Educational. Not investment advice.*

In [ ]:
# ---- Your inputs (the only cell you need to edit) ----
S0     = 1202.0     # stock price today (Rs)
K      = 1200.0     # strike price
DAYS   = 91         # days until expiry
r      = 0.0557     # interest rate per year (5.57% is written 0.0557)
sigma  = 0.2105     # volatility per year (21.05% is written 0.2105)
STEPS  = 3          # steps in the tree. 3 lets you see every number. Try 500 later.
OPTION = "call"     # "call" or "put"
MARKET_PRICE = None # optional: the option's real market price, to compare with the model

import math
T = DAYS / 365      # time in years

## Step 1: how big is one up move and one down move?
Cut the time into small steps of length `dt`. In each step the stock does only two things:
go **up** by a factor `u`, or go **down** by a factor `d`.

* `u = e^(sigma * sqrt(dt))` and `d = 1 / u`  (the bigger the volatility, the bigger the moves)
* `p` is the **risk-neutral probability**. It is *not* the real chance of going up. It is the weight that
  makes the option price consistent with "no free lunch" (nobody can earn risk-free money).

In [ ]:
dt = T / STEPS
u = math.exp(sigma * math.sqrt(dt))
d = 1 / u
p = (math.exp(r * dt) - d) / (u - d)
print(f"dt = {dt:.4f} years    u = {u:.4f}    d = {d:.4f}    p = {p:.4f}")

## Step 2: build the tree of stock prices (going forward)
After `j` steps with `i` of them up, the price is `S0 * u^i * d^(j-i)`.

In [ ]:
stock = [[S0 * u**i * d**(j - i) for i in range(j + 1)] for j in range(STEPS + 1)]

def show(tree, title):
    print(title)
    for j, row in enumerate(tree):
        print(f"  step {j}:  " + "  ".join(f"{x:9.2f}" for x in row))

if STEPS <= 6:
    show(stock, "Stock prices (left = every step down, right = every step up):")
else:
    print(f"{STEPS} steps is too big to print. Last step goes from {stock[-1][0]:.2f} to {stock[-1][-1]:.2f}.")

## Step 3: what is the option worth at expiry?
A call is worth `max(price - strike, 0)`. A put is worth `max(strike - price, 0)`.
At the last step we know these values for certain.

In [ ]:
def payoff(price):
    return max(price - K, 0) if OPTION == "call" else max(K - price, 0)

value = [[0.0] * (j + 1) for j in range(STEPS + 1)]
value[STEPS] = [payoff(x) for x in stock[STEPS]]
if STEPS <= 6:
    show([value[STEPS]], "Option value at expiry:")

## Step 4: work backwards to today
Each earlier node = the **discounted average** of the two nodes after it, using `p`:

`value = e^(-r*dt) * ( p * value_if_up + (1 - p) * value_if_down )`

Keep stepping back until you reach today. That one number is the option price.

In [ ]:
for j in range(STEPS - 1, -1, -1):
    for i in range(j + 1):
        value[j][i] = math.exp(-r * dt) * (p * value[j + 1][i + 1] + (1 - p) * value[j + 1][i])

if STEPS <= 6:
    show(value, "Option values (the top-left number, step 0, is today's price):")
print(f"\nBinomial price with {STEPS} steps: Rs {value[0][0]:.2f} per share")

## Step 5: more steps = a better answer
The same four steps, packed into a short function so we can try many step counts.
**Black-Scholes** is the exact answer a tree reaches with endlessly many steps (for European options).
NSE stock and index options are European.

In [ ]:
def binomial_price(steps):
    dt = T / steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp(r * dt) - d) / (u - d)
    values = [payoff(S0 * u**i * d**(steps - i)) for i in range(steps + 1)]      # value at expiry
    for j in range(steps - 1, -1, -1):                                          # step back one level at a time
        values = [math.exp(-r * dt) * (p * values[i + 1] + (1 - p) * values[i]) for i in range(j + 1)]
    return values[0]

def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def black_scholes():
    d1 = (math.log(S0 / K) + (r + sigma**2 / 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if OPTION == "call":
        return S0 * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S0 * norm_cdf(-d1)

bs = black_scholes()
print(f"Black-Scholes: Rs {bs:.4f}\n")
for n in (3, 10, 50, 100, 500, 1000):
    price = binomial_price(n)
    print(f"{n:5d} steps: Rs {price:8.4f}   (difference from Black-Scholes {price - bs:+.4f})")

## Step 6 (optional): compare with the real market price
Type the option's price from your broker into `MARKET_PRICE` in the inputs cell, then run this cell.
Use the middle of bid and ask if you can. "Cheap" or "expensive" only means *compared with the volatility you typed*.

In [ ]:
model = binomial_price(500)
if MARKET_PRICE is None:
    print(f"Model price (500 steps): Rs {model:.2f}")
    print("Type the option's real price into MARKET_PRICE (first cell) to compare.")
else:
    gap = model - MARKET_PRICE
    print(f"Model price : Rs {model:.2f}")
    print(f"Market price: Rs {MARKET_PRICE:.2f}")
    print(f"Model - market = Rs {gap:+.2f} per share")
    print("Model is higher: the option looks CHEAP at your volatility." if gap > 0
          else "Model is lower: the option looks EXPENSIVE at your volatility.")
    print("Small gaps (a few rupees) are normal: bid-ask gap, costs and your volatility guess all matter.")